**Install Required Libraries**

In [1]:
# Install necessary libraries
!pip install catboost lightgbm xgboost autogluon shap dask[dataframe] scikit-optimize

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of dask-expr to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 8.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take

**Import Libraries**

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from lightgbm import LGBMRegressor
from autogluon.tabular import TabularPredictor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import shap
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
from tqdm import tqdm

**Load and Preprocess the Dataset**

In [2]:
# Load and preprocess the dataset
data = pd.read_csv('/content/Final_Cleaned_Voyage_Data.csv', parse_dates=['DepartureTime', 'ArrivalTime'], infer_datetime_format=True)

# Convert datetimes and calculate duration
data['DepartureTime'] = pd.to_datetime(data['DepartureTime'], format="%m/%d/%Y %H:%M")
data['ArrivalTime'] = pd.to_datetime(data['ArrivalTime'], format="%m/%d/%Y %H:%M")
data['TravelTime_seconds'] = (data['ArrivalTime'] - data['DepartureTime']).dt.total_seconds()

# Extract departure hour
data['DepartureHour'] = data['DepartureTime'].dt.hour

<ipython-input-2-cbb3cf3109e9>:2: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  data = pd.read_csv('/content/Final_Cleaned_Voyage_Data.csv', parse_dates=['DepartureTime', 'ArrivalTime'], infer_datetime_format=True)


**Define Features and Target Variable**

In [3]:
# Define features and target variable
numeric_features = ['LATd', 'LONd', 'LATa', 'LONa', 'Voyage_Distance_Km', 'draft_km', 'Average_Speed_Kmph',
                    'Std_SpeedKMpH', 'Avg_Course', 'breadthKM', 'DepartureHour']
categorical_features = ['vessel_type']

# Preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # Handle missing values
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Handle missing values
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))  # Sparse=False for better compatibility
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Define features and target variable
X = data[numeric_features + categorical_features]
y = data['TravelTime_seconds']

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocess the data
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

**Hyperparameter Tuning for LightGBM**

In [4]:
# Define the hyperparameter search space for LightGBM
lgb_param_grid = {
    'num_leaves': Integer(32, 64),  # Integer range for num_leaves
    'max_depth': Integer(5, 10),    # Integer range for max_depth
    'learning_rate': Real(0.01, 0.1, prior='log-uniform'),  # Log-uniform range for learning_rate
    'n_estimators': Integer(100, 200)  # Integer range for n_estimators
}

# Initialize the LightGBM model
lgb_model = LGBMRegressor(random_state=42)

# Initialize BayesSearchCV
lgb_bayes_search = BayesSearchCV(
    lgb_model,
    lgb_param_grid,
    n_iter=10,  # Number of iterations
    cv=5,       # 5-fold cross-validation
    scoring='neg_mean_squared_error',  # Scoring metric
    random_state=42,
    n_jobs=-1   # Use all available cores
)

# Fit the BayesSearchCV to the training data
lgb_bayes_search.fit(X_train, y_train)

# Get the best model and its hyperparameters
best_lgb_model = lgb_bayes_search.best_estimator_
lgb_y_pred = best_lgb_model.predict(X_test)

# Print the best hyperparameters
print("Best Hyperparameters for LightGBM (BayesSearchCV):")
print(lgb_bayes_search.best_params_)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001864 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2565
[LightGBM] [Info] Number of data points in the train set: 15015, number of used features: 16
[LightGBM] [Info] Start training from score 313731.456210
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

**Hyperparameter Tuning for AutoGluon**

In [5]:
# Hyperparameter tuning for AutoGluon
train_data_autogluon = pd.concat([pd.DataFrame(X_train, columns=numeric_features + list(preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features))),
                                  y_train.reset_index(drop=True)], axis=1)
ag_model = TabularPredictor(label='TravelTime_seconds').fit(train_data_autogluon, time_limit=1200, presets='best_quality')
ag_y_pred = ag_model.predict(pd.DataFrame(X_test, columns=train_data_autogluon.columns[:-1]))

# Get model info
model_info = ag_model.info()

No path specified. Models will be saved in: "AutogluonModels/ag-20250123_135311"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.11.11
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun 27 21:05:47 UTC 2024
CPU Count:          2
Memory Avail:       10.80 GB / 12.67 GB (85.2%)
Disk Space Avail:   75.61 GB / 107.72 GB (70.2%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be f

(_ray_fit pid=1720) [1000]	valid_set's rmse: 46924.7
(_ray_fit pid=1720) [2000]	valid_set's rmse: 43822 [repeated 2x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(_ray_fit pid=1720) [4000]	valid_set's rmse: 42583.8 [repeated 4x across cluster]
(_ray_fit pid=1720) [6000]	valid_set's rmse: 42199.3 [repeated 4x across cluster]
(_ray_fit pid=1722) [7000]	valid_set's rmse: 44370.8 [repeated 3x across cluster]
(_ray_fit pid=1720) [9000]	valid_set's rmse: 41967 [repeated 3x across cluster]
(_ray_fit pid=2017) [1000]	valid_set's rmse: 45115.3 [repeated 4x across cluster]
(_ray_fit pid=2017) [3000]	valid_set's rmse: 39547.2 [repeated 4x across cluster]
(_ray_fit pid=2017) [5000]	valid_set's rmse: 38524.1 [repeated 4x across cluster]
(_ray_fit pid=2017) [7000]	valid_set's rmse: 38287 [repeated 4x across cluster]
(_

(_ray_fit pid=2023) 	Ran out of time, early stopping on iteration 9808. Best iteration is:
(_ray_fit pid=2023) 	[9793]	valid_set's rmse: 29584.4


(_ray_fit pid=2318) [1000]	valid_set's rmse: 56684.9 [repeated 3x across cluster]
(_ray_fit pid=2323) [3000]	valid_set's rmse: 43026.4 [repeated 4x across cluster]
(_ray_fit pid=2318) [4000]	valid_set's rmse: 50000.1 [repeated 3x across cluster]
(_ray_fit pid=2323) [6000]	valid_set's rmse: 41904.4 [repeated 3x across cluster]
(_ray_fit pid=2323) [8000]	valid_set's rmse: 41729.6 [repeated 4x across cluster]
(_ray_fit pid=2318) [9000]	valid_set's rmse: 49259.1 [repeated 3x across cluster]
(_ray_fit pid=2619) [1000]	valid_set's rmse: 42853 [repeated 3x across cluster]
(_ray_fit pid=2669) [2000]	valid_set's rmse: 37649.2 [repeated 3x across cluster]
(_ray_fit pid=2669) [3000]	valid_set's rmse: 36489.5 [repeated 2x across cluster]
(_ray_fit pid=2669) [5000]	valid_set's rmse: 35737.1 [repeated 4x across cluster]
(_ray_fit pid=2669) [7000]	valid_set's rmse: 35487.6 [repeated 4x across cluster]
(_ray_fit pid=2619) [9000]	valid_set's rmse: 37503.6 [repeated 2x across cluster]
(_ray_fit pid=2669

(_dystack pid=1514) 	-40080.4989	 = Validation score   (-root_mean_squared_error)
(_dystack pid=1514) 	217.93s	 = Training   runtime
(_dystack pid=1514) 	65.02s	 = Validation runtime
(_dystack pid=1514) Fitting model: WeightedEnsemble_L2 ... Training model for up to 293.34s of the 53.85s of remaining time.
(_dystack pid=1514) 	Ensemble Weights: {'LightGBMXT_BAG_L1': 1.0}
(_dystack pid=1514) 	-40080.4989	 = Validation score   (-root_mean_squared_error)
(_dystack pid=1514) 	0.01s	 = Training   runtime
(_dystack pid=1514) 	0.0s	 = Validation runtime
(_dystack pid=1514) Fitting 106 L2 models, fit_strategy="sequential" ...
(_dystack pid=1514) Fitting model: LightGBMXT_BAG_L2 ... Training model for up to 53.82s of the 53.80s of remaining time.
(_dystack pid=1514) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (2 workers, per: cpus=1, gpus=0, memory=0.12%)


...

(_ray_fit pid=3081) 	Ran out of time, early stopping on iteration 2050. Best iteration is: [repeated 2x across cluster]
(_ray_fit pid=3081) 	[1818]	valid_set's rmse: 42769.5 [repeated 2x across cluster]


(_ray_fit pid=3215) [1000]	valid_set's rmse: 41278.3
(_ray_fit pid=3220) [1000]	valid_set's rmse: 35696.8


(_ray_fit pid=3215) 	Ran out of time, early stopping on iteration 1875. Best iteration is: [repeated 2x across cluster]
(_ray_fit pid=3215) 	[1605]	valid_set's rmse: 40714.8 [repeated 2x across cluster]


(_ray_fit pid=3350) [1000]	valid_set's rmse: 39657.6
(_ray_fit pid=3382) [1000]	valid_set's rmse: 46956.9


(_ray_fit pid=3350) 	Ran out of time, early stopping on iteration 1687. Best iteration is: [repeated 2x across cluster]
(_ray_fit pid=3350) 	[1687]	valid_set's rmse: 38787 [repeated 2x across cluster]
(_dystack pid=1514) 	-42495.6971	 = Validation score   (-root_mean_squared_error)
(_dystack pid=1514) 	67.73s	 = Training   runtime
(_dystack pid=1514) 	4.05s	 = Validation runtime
(_dystack pid=1514) Fitting model: WeightedEnsemble_L3 ... Training model for up to 293.34s of the -18.27s of remaining time.
(_dystack pid=1514) 	Ensemble Weights: {'LightGBMXT_BAG_L1': 0.682, 'LightGBMXT_BAG_L2': 0.318}
(_dystack pid=1514) 	-39380.9643	 = Validation score   (-root_mean_squared_error)
(_dystack pid=1514) 	0.02s	 = Training   runtime
(_dystack pid=1514) 	0.0s	 = Validation runtime
(_dystack pid=1514) AutoGluon training complete, total runtime = 311.8s ... Best model: WeightedEnsemble_L3 | Estimated inference throughput: 24.1 rows/s (1669 batch size)
(_dystack pid=1514) TabularPredictor saved. T

**Create Meta-Features and Train Random Forest Meta-Learner**

In [6]:
# Create meta-features from base model predictions (LightGBM and AutoGluon-Tabular)
meta_features = pd.DataFrame({'LightGBM': lgb_y_pred, 'AutoGluon': ag_y_pred})

# Define the hyperparameter search space for Random Forest Meta-Learner
rf_param_grid = {
    'n_estimators': Integer(100, 200),  # Number of trees in the forest
    'max_depth': Integer(5, 20),        # Maximum depth of the tree
    'min_samples_split': Integer(2, 10),  # Minimum number of samples required to split a node
    'min_samples_leaf': Integer(1, 5)   # Minimum number of samples required at each leaf node
}

# Initialize the Random Forest model
rf_model = RandomForestRegressor(random_state=42)

# Initialize BayesSearchCV
rf_bayes_search = BayesSearchCV(
    rf_model,
    rf_param_grid,
    n_iter=10,  # Number of iterations
    cv=5,       # 5-fold cross-validation
    scoring='neg_mean_squared_error',  # Scoring metric
    random_state=42,
    n_jobs=-1   # Use all available cores
)

# Fit the BayesSearchCV to the meta-features and target
rf_bayes_search.fit(meta_features, y_test)

# Get the best model and its hyperparameters
best_rf_model = rf_bayes_search.best_estimator_
rf_y_pred = best_rf_model.predict(meta_features)

# Print the best hyperparameters
print("Best Hyperparameters for Random Forest Meta-Learner (BayesSearchCV):")
print(rf_bayes_search.best_params_)

Best Hyperparameters for Random Forest Meta-Learner (BayesSearchCV):
OrderedDict([('max_depth', 17), ('min_samples_leaf', 3), ('min_samples_split', 6), ('n_estimators', 172)])


**Evaluate Stacking Ensemble Performance**

In [7]:
# Evaluate stacking ensemble performance
ensemble_mae = mean_absolute_error(y_test, rf_y_pred)
ensemble_mse = mean_squared_error(y_test, rf_y_pred)
ensemble_r2 = r2_score(y_test, rf_y_pred)

print("Stacking Ensemble (Random Forest Meta-Learner):")
print("Mean Absolute Error:", ensemble_mae)
print("Mean Squared Error:", ensemble_mse)
print("R-squared:", ensemble_r2)

Stacking Ensemble (Random Forest Meta-Learner):
Mean Absolute Error: 1956.4948725724369
Mean Squared Error: 89216934.77246895
R-squared: 0.9993412019865919


**Cross-Validation for Random Forest Meta-Learner**

In [8]:
# Cross-validation for Random Forest Meta-Learner
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'neg_mean_squared_error': 'neg_mean_squared_error',
           'neg_mean_absolute_error': 'neg_mean_absolute_error',
           'r2': 'r2'}
cv_results_rf = cross_validate(best_rf_model, meta_features, y_test, cv=kf, scoring=scoring)
print("Random Forest Meta-learner cross-validation results:")
print("Mean MAE:", -cv_results_rf['test_neg_mean_absolute_error'].mean())
print("Mean MSE:", -cv_results_rf['test_neg_mean_squared_error'].mean())
print("Mean R2:", cv_results_rf['test_r2'].mean())

Random Forest Meta-learner cross-validation results:
Mean MAE: 3002.0481882969734
Mean MSE: 193962619.91963568
Mean R2: 0.9985498722660662


**Compute MAPE for Models**

In [9]:
# Mean Absolute Percentage Error (MAPE)
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero_indices = y_true != 0
    y_true_non_zero = y_true[non_zero_indices]
    y_pred_non_zero = y_pred[non_zero_indices]
    return np.mean(np.abs((y_true_non_zero - y_pred_non_zero) / y_true_non_zero)) * 100

# Compute MAPE for the Stacking Model
stacking_mape = mean_absolute_percentage_error(y_test, rf_y_pred)
print(f"Stacking Ensemble (Random Forest Meta-Learner) MAPE: {stacking_mape:.2f}%")

# Compute MAPE for individual models
lgb_mape = mean_absolute_percentage_error(y_test, lgb_y_pred)
ag_mape = mean_absolute_percentage_error(y_test, ag_y_pred)

print(f"LightGBM Regressor MAPE: {lgb_mape:.2f}%")
print(f"AutoGluon-Tabular MAPE: {ag_mape:.2f}%")

Stacking Ensemble (Random Forest Meta-Learner) MAPE: 0.32%
LightGBM Regressor MAPE: 1.33%
AutoGluon-Tabular MAPE: 0.52%
